# AI Resume Screening System with Tracing
**Innomatics Research Labs - Data Science Internship (Feb 2026)**  
Task 3: GenAI Assignment

In [9]:
# install required libraries
#!pip install langchain langchain-core langsmith huggingface_hub transformers langchain-community

In [10]:
import os
import json
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

## Step 1: Setup API Keys and LangSmith Tracing

In [11]:
# paste your actual keys here
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN", "")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "resume-screening-task3"

print("Keys set.")
print("LangSmith project:", os.environ["LANGCHAIN_PROJECT"])

Keys set.
LangSmith project: resume-screening-task3


## Step 2: Job Description

In [12]:
job_description = """
Job Title: Data Scientist

Required Skills:
- Python programming
- Machine Learning (scikit-learn, XGBoost)
- Deep Learning (TensorFlow or PyTorch)
- SQL and data manipulation
- Data visualization (Matplotlib, Seaborn, or Tableau)
- Statistics and probability
- NLP experience is a plus
- Cloud platforms (AWS, GCP, or Azure)

Experience Required: 2+ years in a data science or analytics role
Tools: Python, SQL, Jupyter Notebook, Git, scikit-learn, TensorFlow
"""

print("Job description loaded.")

Job description loaded.


## Step 3: Define the Three Resumes

In [13]:
# Strong candidate
resume_strong = """
Name: Aisha Khan
Experience: 3 years as Data Scientist at a fintech startup

Skills:
- Python (pandas, numpy, scikit-learn, XGBoost)
- Deep Learning with TensorFlow and Keras
- SQL - advanced queries, joins, window functions
- NLP - text classification and sentiment analysis
- Data visualization with Tableau and Seaborn
- Deployed models on AWS SageMaker
- Git and version control

Projects:
- Customer churn prediction model (XGBoost) - 92% accuracy
- NLP pipeline for ticket classification
- Tableau dashboards for business reporting
"""

# Average candidate
resume_average = """
Name: Ravi Sharma
Experience: 1.5 years as Junior Data Analyst

Skills:
- Python (pandas, matplotlib)
- Basic machine learning with scikit-learn
- SQL - basic queries and joins
- Excel and Google Sheets
- Jupyter Notebook

Projects:
- EDA on sales data
- Simple linear regression model for price prediction
- Basic visualizations for monthly reports
"""

# Weak candidate
resume_weak = """
Name: Priya Mehta
Experience: Fresher - recently completed BCA degree

Skills:
- Basic Python (loops, functions)
- HTML and CSS
- Microsoft Office (Word, Excel)
- Data entry

Projects:
- Library management system in Java
- Simple website using HTML and CSS
"""

print("All 3 resumes loaded.")

All 3 resumes loaded.


## Step 4: Create Prompts

In [14]:
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""You are a resume parser. Only extract what is written. Do NOT add anything.

Resume:
{resume}

List:
1. Skills
2. Years of experience
3. Tools used

Answer:"""
)

matching_prompt = PromptTemplate(
    input_variables=["extracted_info", "job_description"],
    template="""Compare the candidate profile with the job description.

Candidate Profile:
{extracted_info}

Job Description:
{job_description}

List:
- Matching skills
- Missing skills
- Experience match (yes/no with reason)

Answer:"""
)

scoring_prompt = PromptTemplate(
    input_variables=["match_result", "job_description"],
    template="""Based on the match below, give a score from 0 to 100.

Match Analysis:
{match_result}

Job Description:
{job_description}

Rules:
- 80 to 100 if most skills match and experience is enough
- 50 to 79 if partial match
- 0 to 49 if poor match

Reply in this exact format:
Score: <number>
Reason: <one sentence>

Answer:"""
)

explanation_prompt = PromptTemplate(
    input_variables=["score_result", "match_result"],
    template="""Write a short recruiter-style summary for this candidate.

Score and Reason:
{score_result}

Match Analysis:
{match_result}

Write 2-3 sentences. Be honest. Do not assume anything not mentioned.

Answer:"""
)

print("All prompts created.")

All prompts created.


## Step 5: Setup LLM and Build Chains

Using a Hugging Face local model (`gpt2`) through Transformers + LangChain pipeline.

In [15]:
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id)
pipeline_obj = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False,
    truncation=True,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=pipeline_obj)
parser = StrOutputParser()

extraction_chain = extraction_prompt | llm | parser
matching_chain = matching_prompt | llm | parser
scoring_chain = scoring_prompt | llm | parser
explanation_chain = explanation_prompt | llm | parser

print("Hugging Face chains ready.")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9474.75it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Hugging Face chains ready.


## Step 6: Build the Screening Pipeline

In [16]:
def screen_resume(resume_text, jd_text, candidate_label="Candidate"):
    print(f"\n--- Running pipeline for: {candidate_label} ---")

    # step 1: extract skills from resume
    extracted = extraction_chain.invoke({"resume": resume_text})
    print("\nExtracted Info:")
    print(extracted)

    # step 2: match with job description
    match_result = matching_chain.invoke({
        "extracted_info": extracted,
        "job_description": jd_text
    })
    print("\nMatch Result:")
    print(match_result)

    # step 3: score the candidate
    score_result = scoring_chain.invoke({
        "match_result": match_result,
        "job_description": jd_text
    })
    print("\nScore:")
    print(score_result)

    # step 4: explain the result
    explanation = explanation_chain.invoke({
        "score_result": score_result,
        "match_result": match_result
    })
    print("\nFinal Explanation:")
    print(explanation)

    print("\n" + "="*60)

    return {
        "candidate": candidate_label,
        "extracted": extracted,
        "match": match_result,
        "score": score_result,
        "explanation": explanation
    }

print("Pipeline function defined.")

Pipeline function defined.


## Step 7: Run All 3 Resumes

Each run is automatically traced in LangSmith.

In [17]:
# Run 1 - Strong candidate
result_strong = screen_resume(resume_strong, job_description, candidate_label="Strong Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Strong Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


- Python (pandas, numpy, scikit-learn, XGBoost)

- SQL - advanced queries, joins, window functions- NLP - text classification and sentiment analysis- Data visualization with Tableau and Seaborn- Deployed models on AWS SageMaker- Git and version control- Git and version control

Reachability:

- NLP is an open source platform with high-level API and an open source API that can be used for both small and large companies, as well as large and small companies.

-


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:


- "Why are you a data scientist? I'm not a data scientist. I'm a data scientist who knows what I'm doing and what's the best way to do it. I'm not a data scientist with a big passion for statistics like statistics have been for a long time, but I'm a data scientist with a big passion for data science and analytics."

- "I know the best way to do data science is to write code. I don't know the best way to do data science is to read code. I'm a data scientist who knows the best way


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:


This is the main question in our question and answer session.

This is a question about data science, you can answer it in any order you want and we will answer it in the order below.

- What is your career?

- What are your hobbies or experiences?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

-

Final Explanation:


- If you want to say, "Well, I've read about this, but I didn't know how to write code." you have a problem on your hands.

- If you want to say, "I'm just an open source developer, I don't know how to write code and I don't know how to write data, and I've read about that, but I didn't know how to write data." you have a problem on your hands.

- If you want to say, "I've read about this, but I didn't know how to



In [18]:
# Run 2 - Average candidate
result_average = screen_resume(resume_average, job_description, candidate_label="Average Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Average Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


- SQL

- Simple linear regression

- SQL: Python (pandas, matplotlib)

- Basic machine learning with scikit-learn

- SQL - basic queries and joins

- Excel and Google Sheets

- Jupyter Notebook

Projects:

- EDA on sales data

- Simple linear regression model for price prediction

- Basic visualizations for monthly reports


2. Data

3. Tools

4. Project

5. Answer

6. Questions


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:


This job offers a combination of advanced data analysis, deep learning, and machine learning skills. You will need to be able to use Python or SQL to create and execute these tasks. You'll have access to the Python client, and the TensorFlow client.


2.0.1 - Job Description


We are excited to announce that we have moved to a new site: http://jobs.tensorflow.com.

Job Description:

Job Title: Data Processing Engineer

Required Skills:

- Python programming

- Machine learning (sci


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:
 <two sentences>

You can also submit questions and comments. Send a question or comment by using the form below (optional comments are not required)

Email: info@tensorflow.com

Website: http://jobs.tensorflow.com

Job Type: Salesforce

Required Skills:

- Python programming

- Machine learning (scikit)

- Deep Learning (TensorFlow or PyTorch)

- SQL and data manipulation

- Data visualization (Matplotlib, Seaborn, or Tableau)

Final Explanation:


- Python programming

- Machine learning (sci

Use Python 2)

- SQL and data manipulation

- Data visualization (Matplotlib, Seaborn, or Tableau)

Match Analysis:


This job offers a combination of advanced data analysis, deep learning, and machine learning skills. You will need to be able to use Python or SQL to create and execute these tasks. You'll have access to the Python client, and the TensorFlow client.


2.0.0 - Job Description


Job Title: Database Architect




In [19]:
# Run 3 - Weak candidate
result_weak = screen_resume(resume_weak, job_description, candidate_label="Weak Candidate")

Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Running pipeline for: Weak Candidate ---


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Extracted Info:


I've seen people write resumes from this background. I don't have any experience in this field. What I can say is that you can use your resume to create your own resume, or at least your project resume. You can start with a list of skills and work on your own.

Resume:

Name: Agnieszka Zarek

Experience: Resume Language (English, German, Spanish, Português)

Skills:

- PHP

- PHP

- Java

- SQL

-


Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Match Result:


- I'm a Data Scientist

- I'm a Data Scientist

Candidate Profile:

I've seen people write resumes from this background. I don't have any experience in this field. What I can say is that you can use your resume to create your own resume, or at least your project resume. You can start with a list of skills and work on your own.

Resume:

Name: Tanya Joly

Experience: Resume Language (English, German, Spanish, Português)

Skills:



Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Score:
 <one sentence>

Reason: <one sentence>

Reply in this exact format:

Cancellation:

Confirmation Date:

Date of Application:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Confirmation Date:

Conf

Final Explanation:


Tanya Joly

Assistant Director

I think that I'm not qualified to be a position manager on this position. I've worked as Assistant Director for several years, and I am currently an Assistant Director for the New York State Department of Labor. I am also one of the very few people who can speak English fluently. I'm currently working at a data science company in New York State (NYS-DOT) and I am already an Associate Dean, and I have a degree in Human Resources.

I am also currently an Associate Dean, and I have



## Step 8: Summary of All Results

In [20]:
print("SCREENING SUMMARY")
print("="*60)

for result in [result_strong, result_average, result_weak]:
    print(f"\nCandidate: {result['candidate']}")
    print(f"Score: {result['score']}")
    print(f"Explanation: {result['explanation']}")
    print("-"*60)

SCREENING SUMMARY

Candidate: Strong Candidate
Score: 

This is the main question in our question and answer session.

This is a question about data science, you can answer it in any order you want and we will answer it in the order below.

- What is your career?

- What are your hobbies or experiences?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

- What are your hobbies?

-
Explanation: 

- If you want to say, "Well, I've read about this, but I didn't know how to write code." you have a problem on your hands.

- If you want to say, "I'm just an open source developer, I don't know how to write code and I don't know how to write data, and I've read about that, but I didn't know how to write data." you have a problem on your hands.

- If you want to say, "I've read about this, but I didn't know how to
------------------------------------------------------------

Candidate: Aver

## Step 9: LangSmith Tracing

Go to https://smith.langchain.com 

## Bonus: Structured JSON Output

In [21]:
def to_json_summary(result):
    return {
        "candidate": result["candidate"],
        "fit_score": result["score"],
        "explanation": result["explanation"]
    }

summary_json = [
    to_json_summary(result_strong),
    to_json_summary(result_average),
    to_json_summary(result_weak)
]

print(json.dumps(summary_json, indent=2))

[
  {
    "candidate": "Strong Candidate",
    "fit_score": "\n\nThis is the main question in our question and answer session.\n\nThis is a question about data science, you can answer it in any order you want and we will answer it in the order below.\n\n- What is your career?\n\n- What are your hobbies or experiences?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n- What are your hobbies?\n\n-",
    "explanation": "\n\n- If you want to say, \"Well, I've read about this, but I didn't know how to write code.\" you have a problem on your hands.\n\n- If you want to say, \"I'm just an open source developer, I don't know how to write code and I don't know how to write data, and I've read about that, but I didn't know how to write data.\" you have a problem on your hands.\n\n- If you want to say, \"I've read about this, but I didn't know how to"
  },
  {
    "candidate":